# MLP Baseline — Creativity Score Prediction
---
This notebook trains a simple Multi-Layer Perceptron on the **5 rubric features** to predict the composite creativity score.

**Architecture:** `5 features → 64 → 32 → 1 (score)`

⚠️ **Before running:** Upload `train.parquet`, `val.parquet`, and `test.parquet` to your Google Drive under a folder called `AICreativityJudge/data/`.


In [ ]:
# ── Install & Imports ──
!pip install -q pandas pyarrow scikit-learn torch

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import time

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/AICreativityJudge/data'
print("Setup complete.")


In [ ]:
# ── Load Splits ──
train_df = pd.read_parquet(f'{DATA_DIR}/train.parquet')
val_df   = pd.read_parquet(f'{DATA_DIR}/val.parquet')
test_df  = pd.read_parquet(f'{DATA_DIR}/test.parquet')

FEATURE_COLS = [
    'lexical_richness', 'syntactic_complexity',
    'novelty', 'imagery', 'narrative_dynamics'
]
TARGET_COL = 'composite_score'

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print(f"\nFeature stats (train):")
print(train_df[FEATURE_COLS].describe().round(2))


In [ ]:
# ── Prepare PyTorch Tensors ──
def df_to_tensors(df):
    X = torch.tensor(df[FEATURE_COLS].values, dtype=torch.float32)
    y = torch.tensor(df[TARGET_COL].values, dtype=torch.float32).unsqueeze(1)
    return X, y

X_train, y_train = df_to_tensors(train_df)
X_val, y_val     = df_to_tensors(val_df)
X_test, y_test   = df_to_tensors(test_df)

BATCH_SIZE = 256
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, y_val), batch_size=BATCH_SIZE)

print(f"Input shape: {X_train.shape}  Target shape: {y_train.shape}")


In [ ]:
# ── MLP Model ──
class CreativityMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CreativityMLP().to(device)
print(f"Device: {device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(model)


In [ ]:
# ── Train ──
EPOCHS = 50
LR = 1e-3

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

train_losses = []
val_losses = []
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    # Train
    model.train()
    epoch_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(xb)
    train_loss = epoch_loss / len(X_train)
    train_losses.append(train_loss)

    # Validate
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            val_loss += criterion(pred, yb).item() * len(xb)
    val_loss /= len(X_val)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), f'{DATA_DIR}/mlp_best.pt')

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{EPOCHS}  train_mse={train_loss:.4f}  val_mse={val_loss:.4f}  lr={optimizer.param_groups[0]['lr']:.6f}")

print(f"\nBest val MSE: {best_val_loss:.4f}")


In [ ]:
# ── Training Curves ──
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train MSE', color='#00d287')
plt.plot(val_losses, label='Val MSE', color='#00e5ff')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.title('MLP Training Curves')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/mlp_training_curves.png', dpi=150)
plt.show()


In [ ]:
# ── Evaluate on Test Set ──
model.load_state_dict(torch.load(f'{DATA_DIR}/mlp_best.pt'))
model.eval()

with torch.no_grad():
    y_pred = model(X_test.to(device)).cpu().numpy().flatten()
    y_true = y_test.numpy().flatten()

mse = mean_squared_error(y_true, y_pred)
mae = mean_absolute_error(y_true, y_pred)
r2  = r2_score(y_true, y_pred)

print("=" * 50)
print("  MLP BASELINE — TEST SET RESULTS")
print("=" * 50)
print(f"  MSE:  {mse:.4f}")
print(f"  RMSE: {np.sqrt(mse):.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  R²:   {r2:.4f}")
print("=" * 50)

# Scatter plot
plt.figure(figsize=(8, 8))
plt.scatter(y_true, y_pred, alpha=0.05, s=5, color='#00d287')
plt.plot([0, 10], [0, 10], 'r--', alpha=0.7, label='Perfect')
plt.xlabel('True Composite Score')
plt.ylabel('Predicted Composite Score')
plt.title(f'MLP Baseline: Predicted vs True (R²={r2:.3f})')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/mlp_scatter.png', dpi=150)
plt.show()
